# core

> fetching , github, arxiv, url2md and more

core is the fetch-and-read layer. `fetch()` handles any URL from a simple static page to a JavaScript-rendered SPA;
`to_md()` strips HTML to clean markdown. The module also covers pagination across JSON APIs, reading arXiv papers and
 YouTube transcripts, and cloning GitHub repos.

In [ ]:
#| default_exp core

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import json
from fastcore.all import Path,L,timed_cache,globtastic,parallel,run,first,AttrDict,ifnone,fdelegates,bind,patch,setattrs
from fastcore.xdg import xdg_cache_home
from diskcache import Index
import re, shutil, time, os, base64
from contextlib import contextmanager, nullcontext
import lxml.etree as etree
from scrapling import Selector
from scrapling.engines.toolbelt.custom import Response
from scrapling.fetchers import Fetcher, StealthyFetcher, DynamicFetcher
from urllib.parse import urlparse, urlencode
import yt_dlp

In [ ]:
#| export
def fossick_cache(path=None):
    "Get cache path for `name` (e.g. 'arxiv' or 'fetch')"
    p = xdg_cache_home()/'.fossick'
    if path: p /= path
    p.parent.mkdir(parents=True, exist_ok=True)
    return p

In [ ]:
#| export
def _arun(coro) -> any:
	'Run an async coroutine from sync code, even if already inside an event loop'
	import asyncio
	try: asyncio.get_running_loop()
	except RuntimeError: return L(asyncio.run(coro))
	import concurrent.futures
	with concurrent.futures.ThreadPoolExecutor() as pool: return L(pool.submit(asyncio.run, coro).result())

@patch
def mk_bytes(self:Path, data, mode=511, uid=-1, gid=-1):
	"Make all parent dirs of `self`, and write `data`"
	self.parent.mkdir(exist_ok=True, parents=True, mode=mode)
	self.write_bytes(data)
	if uid!=-1 or gid!=-1: os.chown(self, uid, gid)

## Web Fetching

`fetch()` returns a dict with `url`, `status`, `html`, `data` (parsed JSON when the response is JSON), and `xhr`
(captured network calls when `capture_xhr=True`). `to_md()` produces clean markdown, optionally extracting just the
element matched by a CSS selector.

In [ ]:
#| export
def _wrap_md(text, tag): return f'\n<{tag}>\n{text}\n</{tag}>\n' if text else ''

def _html(res_or_html):
    'Extract HTML string from a Page dict or return input if already a string'
    if isinstance(res_or_html, str): return res_or_html
    elif isinstance(res_or_html, Response):
	    return res_or_html.html if getattr(res_or_html,'html', None) is not None else res_or_html.html_content
    return None

def html2md(s:str, ignore_links=True, ignore_images=False):
    'Convert `s` from HTML to markdown'
    import html2text
    from readability import Document
    o = html2text.HTML2Text(bodywidth=5000)
    o.ignore_links, o.mark_code, o.ignore_images = ignore_links, True, ignore_images
    return o.handle(Document(s).summary(keep_all_images=True)) if s.strip() else ''

def to_md(res_or_html,  # Page dict (from fetch/crawl) or raw HTML string
          sel:str=None,  # CSS selector to extract before conversion; returns '' if no match
          multi:bool=False,  # Return all selector matches joined
          wrap_tag:str=None,  # Wrap each multi-result in <wrap_tag>...</wrap_tag>; only used when multi=True
          ignore_links:bool=True,
          ignore_images:bool=False,
          ) -> str:
    'Convert a Page dict or HTML string to clean markdown'
    html = _html(res_or_html)
    _md = lambda h: html2md(str(h), ignore_links, ignore_images)
    if sel:
	    els = Selector(html).css(sel)
	    mds = L(els).map(lambda m: _md(m.get()))
	    if multi: return '\n'.join(mds.map(lambda m: _wrap_md(m, wrap_tag) if wrap_tag else m))
	    html = str(mds[0]) if mds else ''
    return _md(html)

In [ ]:
#| export
def http_page(url, method='GET', payload=None, verify=False, **kw) -> Response:
	kw = dict(verify=verify) | kw
	if method.upper() == 'POST': return Fetcher.post(url, json=payload, **kw)
	if method.upper() == 'PUT': return Fetcher.put(url, json=payload, **kw)
	if payload: kw['params'] = payload
	if method.upper() == 'DELETE': return Fetcher.delete(url, **kw)
	if method.upper() == 'GET': return Fetcher.get(url, **kw)
	else: raise ValueError(f"Unsupported method: {method}")

def get_page(url, method='GET', payload=None, heavy=False, stealthy=False, session=False, **kw) -> Response:
	"Fetch `url`. `session=True` (or a port int) drives the persistent debug Chrome via CDP, reusing its logged-in cookies."
	assert method.upper() in ('GET', 'POST'), 'Only GET and POST methods are supported'
	if session:
		from fossick.cdp import cdp_ws, cdp_cookies
		port = 9223 if session is True else int(session)
		kw.setdefault('cdp_url', cdp_ws(port))
		if (ck := cdp_cookies(url, port=port)): kw.setdefault('cookies', ck)
		kw.pop('verify', None)
		if not stealthy: heavy = True
	if not (heavy or stealthy): return http_page(url, method, payload, **kw)
	f = DynamicFetcher.async_fetch if heavy else StealthyFetcher.async_fetch
	return _arun(f(url, **kw))[0]

http_get = bind(get_page, method='GET', verify=True, timeout=30)
http_post = bind(get_page, method='POST', timeout=30)

In [ ]:
r1=get_page('https://httpbin.org/get', verify=False)
r1.__dict__.keys()

[2026-07-22 12:21:04] INFO: Fetched (200) <GET https://httpbin.org/get> (referer: https://www.google.com/)


dict_keys(['status', 'reason', 'cookies', 'headers', 'request_headers', 'history', 'meta', 'request', 'captured_xhr'])

In [ ]:
from fastcore.test import test_fail

In [ ]:
try: get_page('https://httpbin.org/put', method='PUT', verify=False)
except AssertionError as e: test_fail(str(e) == 'Only GET and POST methods are supported')

In [ ]:
http_page('https://httpbin.org/put', method='PUT', verify=False).status

[2026-07-22 12:21:10] INFO: Fetched (200) <PUT https://httpbin.org/put> (referer: https://www.google.com/)


200

In [ ]:
#| export
_BLOCK_MARKERS = ('cf-chl', 'just a moment', 'attention required', 'checking your browser',
                  'cf-browser-verification', 'cf-turnstile', 'access denied', 'enable javascript',
                  'g-recaptcha', 'h-captcha', 'captcha-delivery',   # captcha *widgets*, not the bare word 'captcha'
                  "making sure you're not a bot", 'id="anubis')     # Anubis proof-of-work wall

def _blocked(page) -> bool:
    "Heuristic: did this response hit a bot wall / challenge page (so we should escalate)?"
    if page is None: return True
    st = getattr(page, 'status', 200)
    if st in (401, 403, 429, 503, 530): return True
    html = getattr(page, 'html', None) or getattr(page, 'html_content', '') or ''
    low = html.lower()
    if any(m in low for m in _BLOCK_MARKERS): return True
    return st == 200 and len(html.strip()) < 200

def _finish(page, sel=None, capture_xhr=False) -> Response:
    "Attach `.html` (raw or `sel`-extracted) and `.xhr` to a scrapling Response"
    html = page.html_content
    if sel: html = str(el.get()) if (el:=page.css(sel).first) else ''
    xhr = []
    if capture_xhr:
        for x in page.captured_xhr:
            ct = x.headers.get('content-Type', '')
            try: d = x.json()
            except: d = x.text or x.body.decode(errors='replace')
            xhr.append(AttrDict(url=x.url, content_type=ct, data=d))
    setattr(page, 'xhr', xhr)
    setattr(page, 'html', html)
    return page

_AUTO_TIERS = (dict(), dict(heavy=True), dict(stealthy=True), dict(stealthy=True, session=True))

def _fetch_auto(url, sel=None, tiers=_AUTO_TIERS, **kw) -> Response:
    "Escalate plain -> heavy -> stealthy -> stealthy+debug-Chrome until a non-blocked response; sets `.tier`."
    page = None
    for opts in tiers:
        try: page = fetch(url, sel=sel, **opts, **kw)
        except Exception: page = None; continue
        if not _blocked(page):
            setattr(page, 'tier', '+'.join(k for k, v in opts.items() if v) or 'plain')
            return page
    if page is not None: setattr(page, 'tier', 'blocked')
    return page

def fetch(url:str,                # URL to fetch
          sel:str=None,           # CSS selector to extract (None = full page)
          method:str='GET',       # HTTP method; 'POST' sends payload as JSON body
          payload:dict=None,      # POST body (JSON) or GET query params
          heavy:bool=False,       # Full JS rendering via headless browser
          stealthy:bool=False,    # Anti-bot stealth fetcher (Cloudflare etc.)
          capture_xhr:bool=False, # Capture XHR calls made by the page (only works with heavy or stealthy)
          session:bool=False,     # Route through the persistent debug Chrome (reuses its logged-in cookies); True or a port int
          auto:bool=False,        # Auto-escalate plain->heavy->stealthy->session on bot-block detection
          **kw,                   # Extra kwargs passed to scrapling (e.g. verify, headers)
         ) -> Response:
    'Fetch `url`, return a Response with `.html` (raw or `sel`-extracted), `.xhr`, `.status`.'
    if auto: return _fetch_auto(url, sel=sel, method=method, payload=payload, capture_xhr=capture_xhr, **kw)
    heavy = heavy or capture_xhr
    if capture_xhr: kw.setdefault('capture_xhr', '.*')
    page = get_page(url, method=method, payload=payload, heavy=heavy, stealthy=stealthy, session=session, **kw)
    return _finish(page, sel, capture_xhr)

@contextmanager
def browser_session(stealthy:bool=False, headless:bool=True, **init_kw):
    "Open ONE browser and yield a `fetch(url, sel=None, **)` func that reuses it across calls (skips per-URL relaunch)."
    from scrapling.fetchers import AsyncDynamicSession, AsyncStealthySession
    from fossick.cdp import syncy  # drive the async session on the one persistent background loop
    init_kw.pop('verify', None)
    Sess = AsyncStealthySession if stealthy else AsyncDynamicSession
    s = Sess(headless=headless, **init_kw)
    syncy(s.__aenter__())
    try:
        def _f(u, sel=None, **fkw): return _finish(syncy(s.fetch(u, **fkw)), sel)
        yield _f
    finally: syncy(s.__aexit__(None, None, None))


### Bot-block detection & escalation

`_blocked()` flags Cloudflare-style bot walls **and** the Anubis proof-of-work wall (HTTP 403/429/503, or a `just a moment` / `checking your browser` / `making sure you're not a bot` interstitial) so `fetch(auto=True)` can walk plain→heavy→stealthy→session and stop at the first tier that isn't blocked, tagging the winner on `.tier`. The cells below hit real protected sites (Cloudflare, Anubis). `fetch(session=True)` and `browser_session()` reuse a real logged-in Chrome and are `eval:false` (they need a running debug Chrome).

In [ ]:
# A site behind Cloudflare: a plain fetch hits the bot wall, and _blocked() flags it for escalation
cf = fetch('https://www.scrapingcourse.com/cloudflare-challenge', verify=False)
assert cf.status == 403, f'expected a Cloudflare 403 challenge, got {cf.status}'
assert _blocked(cf) is True            # fetch(auto=True) escalates from here (heavy -> stealthy -> session)
print('cloudflare:', cf.status, '| blocked?', _blocked(cf))

[2026-07-22 12:21:10] INFO: Fetched (403) <GET https://www.scrapingcourse.com/cloudflare-challenge> (referer: https://www.google.com/)


cloudflare: 403 | blocked? True


In [ ]:
# A site behind Anubis (proof-of-work bot wall): _blocked() now recognizes it, so fetch(auto=True) escalates
an = fetch('https://anubis.techaro.lol/', verify=False)
assert 'making sure' in an.html.lower(), 'expected the Anubis challenge page'
assert _blocked(an) is True            # Anubis markers are in _BLOCK_MARKERS -> auto escalates past this wall
print('anubis wall detected by _blocked?', _blocked(an))

[2026-07-22 12:21:12] INFO: Fetched (200) <GET https://anubis.techaro.lol/> (referer: https://www.google.com/)


anubis wall detected by _blocked? True


In [ ]:
# regression: a normal page that merely *mentions* captcha must NOT read as a bot wall.
# Wikipedia embeds hCaptcha config keys in its page JS -- the word is there, but there's no challenge widget.
wiki = fetch('https://en.wikipedia.org/wiki/Anubis', verify=False)
assert wiki.status == 200 and 'captcha' in wiki.html.lower()   # the word IS present on the page...
assert _blocked(wiki) is False   # ...but _BLOCK_MARKERS matches captcha *widgets* (g-recaptcha/h-captcha/...), not the word

[2026-07-22 12:21:13] INFO: Fetched (200) <GET https://en.wikipedia.org/wiki/Anubis> (referer: https://www.google.com/)


In [ ]:
# fetch(auto=True): walks plain->heavy->stealthy->session, stops at the first unblocked tier, tags .tier
plain = fetch('https://example.com', auto=True, verify=False)
assert plain.status == 200 and plain.tier == 'plain'   # a static page needs no escalation
print('example.com ->', plain.tier)

[2026-07-22 12:21:13] INFO: Fetched (200) <GET https://example.com/> (referer: https://www.google.com/)


example.com -> plain


In [ ]:
#| eval: false
# browser_session(): open ONE browser, reuse it across fetches (skips per-URL relaunch)
with browser_session() as bf:
    a = bf('https://example.com')
    b = bf('https://httpbin.org/html')
assert a.status == 200 and b.status == 200
assert len(a.html) > 0 and len(b.html) > 0

[2026-07-22 12:54:03] INFO: Fetched (200) <GET https://example.com/> (referer: https://www.google.com/)
[2026-07-22 12:54:04] INFO: Fetched (200) <GET https://httpbin.org/html> (referer: https://www.google.com/)


In [ ]:
#| eval: false
# fetch(session=True): route through the persistent debug Chrome, reusing its logged-in cookies
pg = fetch('https://httpbin.org/headers', session=True)
assert pg.status == 200
print(pg.html[:200])

[2026-07-22 12:54:07] INFO: Fetched (200) <GET https://httpbin.org/headers> (referer: https://www.google.com/)


<html><body>{
  "headers": {
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8,application/signed-exchange;v=b3;q=0.7", 
    "Accept-Encod


In [ ]:
#| export
def crawl(start_url:str,               # URL to start from
          sel:str=None,                # CSS selector to extract per page
          follow_sel:str='a[href]',    # CSS selector for links to follow
          same_domain:bool=True,       # Only follow links on same domain
          max_pages:int=10,            # Max pages to visit
          delay:float=0,               # Seconds to wait between requests (polite crawling)
          heavy:bool=False, # Full JS rendering via headless browser
          stealthy:bool=False, # Anti-bot stealth fetcher (Cloudflare etc.)
          reuse:bool=True,  # For heavy/stealthy crawls, keep one browser open across pages
          **kw,                        # Extra kwargs passed to scrapling (e.g. verify, timeout)
         ) -> list:
    'Crawl from `start_url`, following `follow_sel` links, return list of Page dicts'
    bd, v, q, res = urlparse(start_url).netloc, set(), [start_url], []
    use_sess = reuse and (heavy or stealthy)
    ctx = browser_session(stealthy=stealthy, **kw) if use_sess else nullcontext(None)
    with ctx as sess:
        while q and len(v) < max_pages:
            url = q.pop(0)
            if url in v: continue
            v.add(url)
            if delay and res: time.sleep(delay)
            try:
                pg: Response = sess(url) if sess else fetch(url, heavy=heavy, stealthy=stealthy, **kw)
                if pg.status != 200: continue
                if sel: pg.html = str(el.get()) if (el:= pg.css(sel).first) else ''
                res.append(pg)
                for link in pg.css(follow_sel):
                    href = link.attrib.get('href', '')
                    if not href or href.startswith(('#', 'javascript:', 'mailto:')): continue
                    abs_url = pg.urljoin(href)
                    if same_domain and urlparse(abs_url).netloc != bd: continue
                    if abs_url not in v: q.append(abs_url)
            except Exception as ex:
                print(f"Error fetching {url}: {ex}. Continuing crawl.")
                continue
    return res

In [ ]:
#| export
def get_options(page_or_html,  # Page dict (from fetch) or raw HTML string
                sel:str         # CSS selector for the <select> element
               ) -> list:
    "Extract options from a <select> element; returns [{'value': ..., 'text': ...}]"
    return [{'value': el.attrib.get('value', el.text or ''), 'text': (el.text or '').strip()}
            for el in Selector(_html(page_or_html)).css(f'{sel} option')]

def fetch_all(urls:list,           # URLs to fetch
              sel:str=None,         # CSS selector to extract per page (None = full page)
              concurrency:int=8,    # Max parallel fetches
              heavy:bool=False, # Full JS rendering via headless browser
              stealthy:bool=False, # Anti-bot stealth fetcher (Cloudflare etc.)
              **kw                  # Extra kwargs passed to fetch()
             ) -> list:
    "Fetch a list of URLs in parallel; returns Page dicts in the same order as urls"
    return parallel(fetch, urls, sel=sel, heavy=heavy, stealthy=stealthy, threadpool=True, n_workers=concurrency, **kw)

In [ ]:
_sel_html = '''<html><body>
<select id="kanda">
  <option value="1">Balakanda</option>
  <option value="2">Ayodhyakanda</option>
  <option value="3">Aranyakanda</option>
</select>
</body></html>'''

opts = get_options(_sel_html, '#kanda')
assert opts == [{'value': '1', 'text': 'Balakanda'},
                {'value': '2', 'text': 'Ayodhyakanda'},
                {'value': '3', 'text': 'Aranyakanda'}]

### arXiv

In [ ]:
#|export
from liteparse import LiteParse
from pdf_oxide import PdfDocument

In [ ]:
#| export
def get_pdf(url_or_path:str, # URL or local path to PDF
            save_path=None, # optional path to save the PDF (directory or full path)
            **scrapling_kw # extra kwargs passed to scrapling fetcher (e.g. verify, headers, timeout)
            ) -> PdfDocument|None:
    'Fetch PDF from URL and return as PdfDocument'
    if not url_or_path.endswith('.pdf'): return None
    pth = Path(url_or_path) if not url_or_path.startswith('http') else None
    if pth: src = pth.read_bytes() if pth.exists() else None
    else: src = r.body if (r:=get_page(url_or_path, **scrapling_kw)).status == 200 else None
    if not src : return None
    if save_path and isinstance(src, bytes):
	    pth = p/url_or_path.rsplit('/', 1)[-1] if (p:=Path(save_path)).is_dir() else p
	    pth.mk_bytes(src)
    doc = PdfDocument.from_bytes(src) if isinstance(src, bytes) else PdfDocument(src)
    return doc

In [ ]:
import tempfile

In [ ]:
sp = Path(tempfile.mkdtemp())

In [ ]:
sdt = 'https://selfdeterminationtheory.org/SDT/documents/2000_RyanDeci_SDT.pdf'
sdt_p = get_pdf(sdt, save_path=sp, verify=False)

[2026-07-22 12:54:27] INFO: Fetched (200) <GET https://selfdeterminationtheory.org/SDT/documents/2000_RyanDeci_SDT.pdf> (referer: https://www.google.com/)


In [ ]:
sdt_p.to_markdown_all()[:200]

'# Self-Determination  Facilitation of Intrinsic  Social Development,\n\nRichard M. Ryan and  *University of*\n\n*Human beings can be proactive and engaged or, alterna-*  *tively, passive and alienated, la'

In [ ]:
gdr = 'https://selfdeterminationtheory.org/wp-content/uploads/2020/10/1997_GrolnickDeciRyan.pdf'
gcr = get_pdf(gdr, save_path=sp, verify=False)

[2026-07-22 09:48:51] INFO: Fetched (200) <GET https://selfdeterminationtheory.org/wp-content/uploads/2020/10/1997_GrolnickDeciRyan.pdf> (referer: https://www.google.com/)


In [ ]:
gcr.to_markdown_all()[:200]

'> [OCR REQUIRED — page 1]\n> This page is a scanned/rasterised image with no extractable text layer; run OCR to recover its content.\n\n---\n\n> [OCR REQUIRED — page 2]\n> This page is a scanned/rasterised '

In [ ]:
#| export
_arxin = Index(str(fossick_cache('arxiv_cache.db')))
def _fetch_arxiv_meta(arxiv_id:str, **kw):
    "Fetch and parse arxiv metadata for a given ID; result is cached to disk"
    r = http_get(f'https://export.arxiv.org/api/query?id_list={arxiv_id}', **kw)
    if r.status != 200: raise Exception(f"Failed to fetch arxiv data: {r.status}")
    ns = {'a': 'http://www.w3.org/2005/Atom'}
    entry = etree.fromstring(r.body).find('a:entry', ns)
    if entry is None: raise Exception("No paper found")
    def txt(tag): return entry.find(f'a:{tag}', ns).text
    pu = first(L(entry.findall('a:link',ns)).filter(lambda l: l.get('title')=='pdf').map(lambda l: l.get('href')))
    au = L(entry.findall('a:author', ns)).map(lambda a: a.find('a:name', ns).text)
    return dict(title=txt('title').strip(), summary=txt('summary').strip(),
                published=txt('published'), link=txt('id'), pdf_url=pu, authors=au)

In [ ]:
#| export
def read_arxiv(url:str, # arxiv PDF URL, or arxiv abstract URL, or arxiv ID
               save_pdf:bool=True, # if True, saves the downloaded PDF to disk
               save_dir:str='.', # directory in which to save the PDF
               force:bool=False, # if True, forces re-download of PDF even if it exists on disk
               **kw # http_get kwargs (e.g. verify=False to skip SSL verification, headers=... etc.)
               )-> dict:
    "Get paper information from arxiv URL or ID, optionally saving PDF to disk"
    arxiv_id = url.rsplit('/', 1)[-1]
    ver = re.search(r'v(\d+)$', arxiv_id)
    ver_sfx = f'v{ver.group(1)}' if ver else ''
    arxiv_id = re.sub(r'v\d+$', '', arxiv_id)
    pdf_path = Path(save_dir)/f'{arxiv_id}{ver_sfx}.pdf' if save_pdf else None
    if force: _arxin.pop(arxiv_id, None)
    if arxiv_id in _arxin: return _arxin[arxiv_id]
    res = dict(_fetch_arxiv_meta(arxiv_id), **kw)
    if not res['pdf_url']: raise Exception("No PDF URL found in arxiv metadata")
    doc = get_pdf(res['pdf_url'], save_path=pdf_path, force=force, **kw)
    res |= dict(doc=doc)
    _arxin[arxiv_id] = res
    return res

In [ ]:
read_arxiv('https://arxiv.org/abs/2306.14881')['summary'][:200]

'Low-metallicity dwarf galaxies often show no or little CO emission, despite the intense star formation observed in local samples. Both simulations and resolved observations indicate that molecular gas'

In [ ]:
#| export
def _crossref_search(**params):
    "Search Crossref API; returns parsed response dict"
    return fetch(f'https://api.crossref.org/works?{urlencode(params)}', verify=False).json() or {}

_crossref = bind(_crossref_search, **{'select': 'DOI,title', 'rows': 1})

def lookup_doi(title:str  # Paper title to search
               ) -> str|None:
    "Return doi.org URL for first Crossref match on `title`"
    items = _crossref(**{'query.title': title}).get('message', {}).get('items', [])
    doi = items[0].get('DOI') if items else None
    return f"https://doi.org/{doi}" if doi else None

In [ ]:
#| eval: False
lookup_doi('Self-Determination Theory and the Facilitation of Intrinsic Motivation, Social Development, and Well-Being')

[2026-07-22 09:49:08] INFO: Fetched (200) <GET https://api.crossref.org/works?select=DOI%2Ctitle&rows=1&query.title=Self-Determination+Theory+and+the+Facilitation+of+Intrinsic+Motivation%2C+Social+Development%2C+and+Well-Being> (referer: https://www.google.com/)


'https://doi.org/10.1037//0003-066x.55.1.68'

In [ ]:
#| export
from fastcore.nbio import Notebook, mk_cell, new_nb, os

In [ ]:
#| export
def _nb_stem(url):
	"Derive a filesystem-safe notebook stem from a URL"
	path = urlparse(url).path.rsplit('/', 1)[-1].split('.')[0]
	stem = path or urlparse(url).netloc.split('.')[0]
	return re.sub(r'[^\w-]', '_', stem)[:50]

def clean_md(md):
	md = re.sub(r'([a-z])-\n([a-z])', r'\1\2', md)
	md = re.sub(r'([a-z])-\n([A-Z])', r'\1-\2', md)
	md = re.sub(r'^ +', '', md, flags=re.M)
	md = re.sub(r' {2,}', ' ', md)
	return re.sub(r'\n{3,}', '\n\n', md)

def _text_segs(t): return [c.strip() for c in re.split(r'\n(?=#{1,3} )', t) if c.strip()]

def _md2cells(md):
	'Split markdown into cells; [code]...[/code] blocks from html2text become code cells'
	code_re = re.compile(r'\[code\]\n\n(.*?)\n\[/code\]', re.DOTALL)
	segments, last_end = [], 0
	for m in code_re.finditer(md):
		text = md[last_end:m.start()].strip()
		if text: segments += [('markdown', s) for s in _text_segs(text)]
		code = re.sub(r'^    ', '', m.group(1), flags=re.MULTILINE).strip()
		if code: segments += [('code', code)]
		last_end = m.end()
	if text := md[last_end:].strip(): segments += [('markdown', s) for s in _text_segs(text)]
	cells = []
	for i, (kind, content) in enumerate(segments):
		if kind == 'code': cells.append(mk_cell(content, 'code'))
		else: cells.append(mk_cell(content, 'markdown'))
	return cells

def ocr_parse(pdf:PdfDocument, # PdfDocument object
              **kw  # extra kwargs passed to LiteParse (e.g. dpi, num_workers, extract_links)
) -> str:
	lp = LiteParse(ocr_enabled=True, dpi=300, num_workers=4, extract_links=True, **kw)
	md = lp.parse(pdf.to_bytes()).text
	md = clean_md(md)
	return md

@fdelegates(ocr_parse)
def pdf2md(pdf:PdfDocument,  # PdfDocument object
           out_path:str|Path,  # Path to notebook file (used to determine image output dir)
           image_dir='images',  # subdir for extracted images
           preserve_layout=False,  # preserrve layout in pdf-oxide
           ocr_selection:str='auto'  # choosing ocr (auto, off, on). auto uses pdf-oxide's ocr detection
           ) -> str:
	'PdfDocument to markdown; if OCR is required, uses LiteParse for better formatting'
	if ocr_selection == 'on': return ocr_parse(pdf)
	cwd = os.getcwd()
	out_path = Path(out_path).resolve()
	os.chdir(out_path.parent)
	imdir = Path(out_path.stem) / image_dir
	Path(imdir).mkdir(exist_ok=True, parents=True)
	md = pdf.to_markdown_all(preserve_layout, include_images=True, embed_images=False, image_output_dir=str(imdir))
	md = clean_md(md)
	os.chdir(cwd)
	if ocr_selection == 'off': return md
	if '> [OCR REQUIRED' in md.strip(): return ocr_parse(pdf)
	return md

@fdelegates(fetch)
def url2nb(url, # URL to convert (PDF, arxiv, or HTML page)
           nb_path=None, # optional path to save the notebook; defaults to <url-stem>.ipynb
           force=False, # if True, forces re-download and conversion even if notebook exists
           **kwargs # extra kwargs passed to fetch() (e.g. verify=False to skip SSL verification)
)-> Path:
	'Convert any URL (PDF, arxiv, or HTML page) to a Jupyter notebook'
	if url.endswith('.pdf'): return pdf2nb(url, nb_path=nb_path, **kwargs)
	nb_path = Path(nb_path or f'{_nb_stem(url)}.ipynb')
	if nb_path.exists() and not force: return nb_path
	if 'arxiv.org' in url:
		res = read_arxiv(url)
		return pdf2nb(str(res['pdf_path']), nb_path=nb_path)
	md = to_md(fetch(url, **kwargs))
	Notebook(new_nb(cells=_md2cells(md)), path=nb_path).save()
	return nb_path

@fdelegates(fetch)
def pdf2nb(u_or_p,  # URL or local path to PDF
           nb_path=None,  # optional path to save the notebook; defaults to <pdf-stem>.ipynb
           force=False,  # if True, forces re-download and conversion even if notebook exists
           ocr_selection:str='auto',  # choosing ocr (auto, off, on). auto uses pdf-oxide's ocr detection
           **kwargs,  # extra kwargs passed to fetch() (e.g. verify=False to skip SSL verification)
           ) -> Path:
	'Convert PDF to a Jupyter notebook; one markdown cell per page + empty code cell'
	stem = Path(u_or_p).stem if not u_or_p.startswith('http') else u_or_p.rsplit('/', 1)[-1].split('.')[0]
	pth = nb_path or f'{stem}.ipynb'
	nb_path = Path(pth).resolve()
	if nb_path.exists() and not force: return nb_path
	pdf = get_pdf(u_or_p, **kwargs)
	if not pdf: raise Exception("Failed to fetch or parse PDF")
	text = pdf2md(pdf, out_path=nb_path, ocr_selection=ocr_selection)
	Notebook(new_nb(cells=_md2cells(text)), path=nb_path).save()
	return nb_path

In [ ]:
get_pdf('https://papers.neurips.cc/paper/7181-attention-is-all-you-need.pdf', save_path=sp, verify=False)

[2026-07-22 09:49:17] INFO: Fetched (200) <GET https://proceedings.neurips.cc/paper_files/paper/2017/file/3f5ee243547dee91fbd053c1c4a845aa-Paper.pdf> (referer: https://www.google.com/)


PdfDocument(version=1.3)

In [ ]:
#| eval: False
url2nb('https://squiddev.medium.com/continuing-continuations-cps-in-python-47bba90c8d1e', nb_path=sp, verify=False)

Path('/var/folders/kg/9vdw4mdd1fs58svgh4k1qhr09x7dqh/T/tmpizbnoazt')

In [ ]:
#| eval: False
pdf2nb(sdt, nb_path=Path(sp)/'test.ipynb', verify=False)

[2026-07-02 12:47:27] INFO: Fetched (200) <GET https://selfdeterminationtheory.org/SDT/documents/2000_RyanDeci_SDT.pdf> (referer: https://www.google.com/)


Path('/private/var/folders/kg/9vdw4mdd1fs58svgh4k1qhr09x7dqh/T/tmpvlob7i55/test.ipynb')

### GitHub

In [ ]:
#| export
def _gh_ssh(url:str): # GitHub URL, SSH remote, or bare `user/repo`
    'Convert GitHub URL to SSH remote; pass through if already SSH; return None if not GitHub'
    if url.startswith('git@github.com:'): return url
    if m := re.match(r'https://github\.com/([^/]+)/([^/]+)', url):
        return f'git@github.com:{m.group(1)}/{m.group(2)}.git'

def _get_git_repo(gh_ssh:str):
    'Clone or update a GitHub repo to local cache; return Path'
    repo_name = gh_ssh.rsplit('/', 1)[-1].removesuffix('.git')
    repo_dir = fossick_cache('git_clones') / repo_name
    if repo_dir.exists():
        try:
            run(['git', '-C', str(repo_dir), 'fetch'])
            branch = run(['git', '-C', str(repo_dir), 'symbolic-ref', 'refs/remotes/origin/HEAD']).split('/')[-1]
            run(['git', '-C', str(repo_dir), 'reset', '--hard', f'origin/{branch}'])
            return repo_dir
        except IOError: shutil.rmtree(repo_dir)
    repo_dir.parent.mkdir(parents=True, exist_ok=True)
    run(['git', 'clone', gh_ssh, str(repo_dir)])
    return repo_dir

@timed_cache(seconds=3600)
def read_gh_repo(path_or_url:str,  # GitHub URL, SSH address, or local path
                 globs:tuple=None,  # file glob patterns (default: README*, pyproject.toml, *.py)
                 limit:int=None,    # max files to return
                 as_list:bool=False # return list of Paths instead of {path: content} dict
                )-> dict:
    'Read files from a GitHub repo filtered by glob patterns'
    ssh = _gh_ssh(path_or_url)
    repo_dir = _get_git_repo(ssh) if ssh else Path(path_or_url)
    if globs is None: globs = ('README*', 'pyproject.toml', '*.py')
    files = L(p for g in globs for p in globtastic(repo_dir, file_glob=g, func=Path)).unique()
    if limit: files = files[:limit]
    if as_list: return files
    return {str(p): p.read_text(errors='ignore') for p in files}

def read_gh_file(url:str # GitHub blob URL of the file to read
                 ):
	'Read raw contents of a file from its GitHub URL'
	raw_url = re.sub(r'https://github\.com/([^/]+)/([^/]+)/blob/([^/]+)/(.+)',
	                 r'https://raw.githubusercontent.com/\1/\2/refs/heads/\3/\4', url)
	if (r:=http_get(raw_url)).status != 200: raise Exception(f"Failed to fetch {raw_url}: {r.status}")
	return to_md(r)

In [ ]:
read_gh_file('https://github.com/Karthik777/litesearch/blob/main/README.md')[:200]

[2026-07-22 09:49:27] INFO: Fetched (200) <GET https://raw.githubusercontent.com/Karthik777/litesearch/refs/heads/main/README.md> (referer: https://www.google.com/)


'# litesearch > **NB** Reading this on GitHub? The formatted [documentation](https://Karthik777.github.io/litesearch/) is nicer. litesearch stores and searches documents in a single SQLite database. It'

In [ ]:
#| eval: false
list(read_gh_repo('https://github.com/vedicreader/gheasy'))

['/Users/71293/.cache/.fossick/git_clones/gheasy/README.md',
 '/Users/71293/.cache/.fossick/git_clones/gheasy/pyproject.toml',
 '/Users/71293/.cache/.fossick/git_clones/gheasy/gheasy/__init__.py',
 '/Users/71293/.cache/.fossick/git_clones/gheasy/gheasy/_modidx.py',
 '/Users/71293/.cache/.fossick/git_clones/gheasy/gheasy/core.py',
 '/Users/71293/.cache/.fossick/git_clones/gheasy/gheasy/workflow.py']

## API Discovery & Pagination

`find_xhr()` visits a page with a real browser, captures all XHR and fetch calls it makes, and returns those matching a
URL pattern. This surfaces the undocumented JSON endpoints that JavaScript-heavy sites use to load their data.
`paginate_api()` replays one of those captured requests across pages until results are exhausted.

In [ ]:
#| export
def compile_pattern(pattern):
	"Compile pattern as regex; if invalid (e.g. bare glob like *foo*), convert via fnmatch first"
	try: return re.compile(pattern)
	except re.error:
		import fnmatch
		return re.compile(fnmatch.translate(pattern))

def find_xhr(url:str,             # URL to visit with browser
             pattern:str='*',     # Glob or regex pattern to filter captured XHR URLs
             session:bool=False,  # Use the persistent debug Chrome (authenticated) instead of a throwaway browser
             port:int=9223,       # debug Chrome port (only when session=True)
             tail:int=3,          # seconds to keep listening after navigation (only when session=True)
             **kw,                # Extra kwargs passed to fetch (verify, network_idle, etc.)
            ) -> list:
    "Visit `url`, return [{url, content_type, data, capture}] for each XHR/fetch call made. `capture` replays via `replay_xhr`."
    rx = compile_pattern(pattern)
    if session:
        from fossick.cdp import cdp_connect, syncy
        async def _run():
            cdp = await cdp_connect(port=port)
            return await cdp.calls(url=url, pattern=pattern, tail=tail)
        out = []
        for r in syncy(_run()).values():
            rb = r.get('response_body')
            if isinstance(rb, dict):
                rb = base64.b64decode(rb.get('body','')).decode(errors='replace') if rb.get('base64Encoded') else rb.get('body','')
            try: d = json.loads(rb) if rb else None
            except Exception: d = rb
            ct = {k.lower(): v for k, v in (r.get('headers') or {}).items()}.get('content-type', '')
            out.append(AttrDict(url=r['url'], content_type=ct, data=d, capture=r))
        return out
    kw.setdefault('network_idle', True)
    xhr = fetch(url, capture_xhr=True, **kw).xhr
    return [x for x in xhr if rx.search(x.url)]

def replay_xhr(capture:dict,        # a request dict from cdp.calls() or find_xhr(session=True)[i].capture
               data=None,           # override request body (str/dict); defaults to the captured body
               port:int=9223,       # debug Chrome port to pull cookies from
               use_cookies:bool=True, # attach the browser's cookies for this domain (keeps auth)
               **kw,                # Extra kwargs passed to the underlying Fetcher (headers, params, verify)
              ) -> Response:
    "Replay a browser-captured request as a fast plain-HTTP call, authenticated via the debug Chrome's cookies."
    url = capture['url']
    method = (capture.get('method') or 'GET').upper()
    headers = {k: v for k, v in (capture.get('headers') or {}).items()
               if k.lower() not in ('content-length', 'host', 'connection', 'accept-encoding', 'cookie')}
    if use_cookies:
        from fossick.cdp import cdp_cookies
        if (ck := cdp_cookies(url, port=port, as_dict=True)): kw.setdefault('cookies', ck)
    kw.setdefault('headers', headers)
    kw.setdefault('verify', False)
    body = data if data is not None else capture.get('postData')
    if method == 'GET': return Fetcher.get(url, **kw)
    if isinstance(body, str):
        try: body = json.loads(body)
        except: return Fetcher.post(url, data=body, **kw)
    return Fetcher.post(url, json=body, **kw)

In [ ]:
assert compile_pattern('.*products.*').search('https://api.example.com/products?q=1')
assert compile_pattern('*products*').search('https://api.example.com/products?q=1')
assert not compile_pattern('*products*').search('https://api.example.com/search')
assert compile_pattern('.*[Ss]earch.*').search('https://api.example.com/Search')

In [ ]:
#| export
def paginate_api(url:str,                      # API endpoint URL
                 payload:dict=None,            # Request body (POST) or params (GET)
                 page_field:str='pageNumber',  # Payload key to increment for each page
                 size_field:str='pageSize',    # Payload key for page size (detects last page)
                 results_field:str=None,       # Response key with items list (auto-detect if None)
                 method:str='POST',            # HTTP method
                 max_pages:int=10,
                 page_size=24,                 # Page size to request (only used if not in payload)
                 page_start:int=1,             # Starting page number (default 1)
                 save:bool=False,              # If True, saves each page's items to disk
                 save_file:str='{url}_page_{page}.json', # Filepath pattern for saving (only used if save=True)
                 force:bool=False,             # If True, forces re-fetching even if saved file exists
                 **kw,                         # Extra kwargs passed to fetch() (verify, headers, etc.)
                ) -> list:
    "Paginate through a JSON API, collecting all results. Auto-detects the items list in response."
    payload = dict(payload or {})
    page_size = payload.get(size_field, page_size)
    all_items = []
    for page_num in range(page_start, max_pages + page_start):
        sf = save_file.format(url=urlparse(url).netloc, page=page_num)
        if not force and Path(sf).exists():
            print(f"Page {page_num} already saved, skipping fetch")
            with open(sf) as f: all_items.extend(json.load(f)); continue
        if (pg:= fetch(url, method=method, payload={**payload, page_field: page_num}, **kw)).status != 200: break
        try: data = pg.json()
        except: break
        if isinstance(data, list): items = data
        elif results_field:
            raw = data.get(results_field, [])
            items = list(raw.values()) if isinstance(raw, dict) else raw
        else:
            lists = {k: v for k, v in data.items() if isinstance(v, list)}
            items = max(lists.values(), key=len) if lists else []
        if not items: break
        all_items.extend(items)
        if len(items) < page_size: break
        if save: Path(sf).write_text(json.dumps(items, indent=2))
    return all_items

In [ ]:
#| export
def download_files(url:str,
                   pattern:str='*',          # Glob or regex to match download hrefs
                   save_dir:str='.',         # Directory to save files
                   follow:bool=False,        # Crawl linked pages looking for more files
                   follow_sel:str='a[href]', # Selector for page links to follow when crawling
                   file_sel:str='a[href]',   # Selector to scan for download links on each page
                   same_domain:bool=True,
                   max_pages:int=10,
                   delay:float=0,
                   force:bool=False,         # Re-download even if file already exists
                   heavy:bool=False,
                   stealthy:bool=False,
                   **kw,                     # Extra kwargs passed to fetch/http_get (e.g. verify)
                  ) -> list[Path]:
    "Download all hrefs matching pattern from url; optionally crawl linked pages first"
    save_dir = Path(save_dir)
    save_dir.mkdir(parents=True, exist_ok=True)
    rx = compile_pattern(pattern)
    pages = (crawl(url, follow_sel=follow_sel, same_domain=same_domain, max_pages=max_pages, delay=delay,
       heavy=heavy, stealthy=stealthy, **kw) if follow else [fetch(url, heavy=heavy, stealthy=stealthy, **kw)])
    saved, seen = [], set()
    for pg in pages:
        for a in pg.css(file_sel):
            href = a.attrib.get('href', '')
            if not rx.search(href): continue
            abs_url = pg.urljoin(href)
            if abs_url in seen: continue
            seen.add(abs_url)
            fname = save_dir / abs_url.rsplit('/', 1)[-1].split('?')[0]
            if fname.exists() and not force: saved.append(fname); continue
            if (r:=get_page(abs_url, verify=kw.get('verify', False))).status == 200: fname.write_bytes(r.body)
            saved.append(fname)
    return saved

In [ ]:
#| eval: False
jax, fs = 'https://docs.jax.dev/en/latest/notebooks/thinking_in_jax.html', 'a[class="right-next"]'
download_files(jax, follow=True, follow_sel=fs, file_sel='a[href$=".ipynb"]', save_dir=sp, verify=False)

[2026-07-02 12:47:49] INFO: Fetched (200) <GET https://docs.jax.dev/en/latest/notebooks/thinking_in_jax.html> (referer: https://www.google.com/)
[2026-07-02 12:47:49] INFO: Fetched (200) <GET https://docs.jax.dev/en/latest/notebooks/Common_Gotchas_in_JAX.html> (referer: https://www.google.com/)
[2026-07-02 12:47:49] INFO: Fetched (200) <GET https://docs.jax.dev/en/latest/jax-101.html> (referer: https://www.google.com/)
[2026-07-02 12:47:49] INFO: Fetched (200) <GET https://docs.jax.dev/en/latest/jit-compilation.html> (referer: https://www.google.com/)
[2026-07-02 12:47:50] INFO: Fetched (200) <GET https://docs.jax.dev/en/latest/automatic-vectorization.html> (referer: https://www.google.com/)
[2026-07-02 12:47:50] INFO: Fetched (200) <GET https://docs.jax.dev/en/latest/automatic-differentiation.html> (referer: https://www.google.com/)
[2026-07-02 12:47:50] INFO: Fetched (200) <GET https://docs.jax.dev/en/latest/pytrees.html> (referer: https://www.google.com/)
[2026-07-02 12:47:51] INFO:

[Path('/var/folders/kg/9vdw4mdd1fs58svgh4k1qhr09x7dqh/T/tmpvlob7i55/thinking_in_jax.ipynb'),
 Path('/var/folders/kg/9vdw4mdd1fs58svgh4k1qhr09x7dqh/T/tmpvlob7i55/thinking_in_jax.ipynb'),
 Path('/var/folders/kg/9vdw4mdd1fs58svgh4k1qhr09x7dqh/T/tmpvlob7i55/thinking_in_jax.ipynb'),
 Path('/var/folders/kg/9vdw4mdd1fs58svgh4k1qhr09x7dqh/T/tmpvlob7i55/Common_Gotchas_in_JAX.ipynb'),
 Path('/var/folders/kg/9vdw4mdd1fs58svgh4k1qhr09x7dqh/T/tmpvlob7i55/Common_Gotchas_in_JAX.ipynb'),
 Path('/var/folders/kg/9vdw4mdd1fs58svgh4k1qhr09x7dqh/T/tmpvlob7i55/Common_Gotchas_in_JAX.ipynb'),
 Path('/var/folders/kg/9vdw4mdd1fs58svgh4k1qhr09x7dqh/T/tmpvlob7i55/parallel.ipynb'),
 Path('/var/folders/kg/9vdw4mdd1fs58svgh4k1qhr09x7dqh/T/tmpvlob7i55/parallel.ipynb')]

In [ ]:
from fastcore.test import test_eq

In [ ]:
_pages = crawl('https://lego.sankalpa.sh/', max_pages=2, verify=False)
assert isinstance(_pages, list), f"Expected list, got {type(_pages)}"
assert len(_pages) > 0, "Expected at least one page"
assert all(p.status == 200 for p in _pages), "Non-200 pages should be skipped"
assert all(len(p.html) > 0 for p in _pages), "html should be non-empty"
assert len({p.url for p in _pages}) == len(_pages), "url values should be unique"

[2026-07-02 12:49:04] INFO: Fetched (200) <GET https://lego.sankalpa.sh/> (referer: https://www.google.com/)


In [ ]:
#| eval: false
# crawl(reuse=True): a JS-rendered crawl keeps ONE browser open across pages (no per-page relaunch)
_jsp = crawl('https://quotes.toscrape.com/js/', max_pages=2, heavy=True, reuse=True)
assert _jsp and all(p.status == 200 for p in _jsp)
assert all(len(p.html) > 500 for p in _jsp)      # JS actually rendered content into the DOM

In [ ]:
#| eval: false
# Step 1: visit the listing page with a headless browser, capture all XHR/fetch calls
apis = find_xhr('https://www.danmurphys.com.au/list/wine-all', verify=False)

In [ ]:
#| eval: false
# paginate_api: test with JSONPlaceholder (free public REST API, GET-based)
posts = paginate_api(
    'https://jsonplaceholder.typicode.com/posts',
    payload={'_page': 1, '_limit': 5},
    page_field='_page',
    size_field='_limit',
    method='GET',
    verify=False,
)
assert len(posts) >= 5
assert 'title' in posts[0]

`find_xhr(session=True)` captures a page's XHR through the authenticated debug Chrome; each hit carries a `capture` that `replay_xhr()` re-issues as a fast plain-HTTP call using the browser's cookies (no browser relaunch, keeps auth).

In [ ]:
#| eval: false
# find_xhr(session=True) -> replay_xhr: capture a request via the authenticated debug Chrome,
# then replay it as a fast plain-HTTP call using the browser's cookies
hits = find_xhr('https://httpbin.org/json', pattern='.*json', session=True)
assert hits, 'no matching request captured'
assert hits[0].capture, 'each hit should carry a replayable capture'
r = replay_xhr(hits[0].capture)
assert r.status == 200
print(hits[0].url, '->', r.status)

## YouTube

`search_yt()` runs a YouTube search and returns metadata for each video. `read_yt()` fetches the auto-generated English
 captions as plain text, disk-cached by video ID. `download_yt()` saves audio or video to disk.

In [ ]:
#| export
_yt_cache = Index(str(fossick_cache('yt_cache.db')))

def _yt_id(url:str) -> str:
    "Extract 11-char video ID from any YouTube URL or pass through bare ID"
    if m := re.search(r'(?:v=|youtu\.be/|/shorts/|/embed/)([A-Za-z0-9_-]{11})(?![A-Za-z0-9_-])', url): return m.group(1)
    return url

def _parse_vtt(vtt:str) -> str:
    "Strip VTT timing lines and cue tags, deduplicate adjacent lines, return plain text"
    lines = []
    for line in vtt.splitlines():
        if (re.match(r'^\d{2}:\d{2}', line) or '-->' in line
                or re.match(r'^(WEBVTT|Kind:|Language:|NOTE)', line)
                or not line.strip()): continue
        text = re.sub(r'<[^>]+>', '', line).strip()
        if text and (not lines or lines[-1] != text): lines.append(text)
    return ' '.join(lines)

In [ ]:
#| export
def search_yt(q:str, # search query
              n:int=10 # number of results to return
) -> list:
    "Search YouTube; returns L of dicts: id, title, url, duration, view_count, channel, description, thumbnail"
    try:
        with yt_dlp.YoutubeDL({'quiet': True, 'extract_flat': True}) as ydl:
            return ydl.extract_info(f'ytsearch{n}:{q}', download=False).get('entries',[])
    except Exception as e:
        print(f'search_yt error: {e}')
        return []

In [ ]:
#| eval: false
results = search_yt('3blue1brown neural networks', n=3)
assert len(results) >= 1, f"expected results, got {len(results)}"
assert any(kw in results[0]['title'].lower() for kw in ('3blue1brown', 'neural', 'network')), \
    f"unexpected title: {results[0]['title']}"
assert results[0]['url'].startswith('https://www.youtube.com'), f"bad url: {results[0]['url']}"
print(results[0]['title'], '|', results[0]['url'])

But what is a neural network? | Deep learning chapter 1 | https://www.youtube.com/watch?v=aircAruvnKk


In [ ]:
#| export
def read_yt(url:str, # YouTube URL or video ID
            force:bool=False # if True, forces re-fetching even if cached
) -> dict:
    'Fetch YouTube metadata + English transcript (auto-captions); result disk-cached by video ID'
    vid = _yt_id(url)
    if not force and vid in _yt_cache: return _yt_cache[vid]
    with yt_dlp.YoutubeDL({'quiet': True, 'skip_download': True}) as ydl:
        info = ydl.extract_info(f'https://www.youtube.com/watch?v={vid}', download=False)
    caps = info.get('automatic_captions', {})
    en_caps = caps.get('en') or caps.get('en-orig') or []
    vtt_url = next((c['url'] for c in en_caps if c.get('ext') == 'vtt'), None)
    source = ''
    if vtt_url:
        r = http_get(vtt_url)
        if r.status_code == 200: source = _parse_vtt(r.text)
    res = dict(title=info.get('title', ''),
               channel=info.get('channel') or info.get('uploader', ''),
               description=info.get('description', ''),
               duration=info.get('duration'),
               upload_date=info.get('upload_date'),
               source=source)
    _yt_cache[vid] = res
    return res

In [ ]:
meta = read_yt('https://www.youtube.com/watch?v=aircAruvnKk')
assert meta['title'], "title should be non-empty"
assert isinstance(meta['source'], str), "source should be a string"
assert len(meta['source']) > 100, f"transcript too short: {len(meta['source'])} chars"
assert '3blue1brown' in meta['channel'].lower(), f"unexpected channel: {meta['channel']}"
print(f"title: {meta['title']}")
print(f"transcript preview: {meta['source'][:200]}")

In [ ]:
#| export
def download_yt(url:str, # YouTube URL or video ID
                format:str='audio', # 'audio'|'video'|yt-dlp format string
                save_dir:str='.', # directory to save file
                quality:str=None # preferred audio quality (e.g. '192' for 192kbps, only used if format='audio')
) -> Path:
    'Download YouTube media; format=\'audio\'|\'video\'|yt-dlp format string. Returns Path to saved file.'
    save_dir = Path(save_dir)
    save_dir.mkdir(parents=True, exist_ok=True)
    outtmpl = str(save_dir / '%(title)s.%(ext)s')
    if format == 'audio':
        opts = {'quiet': True, 'format': 'bestaudio/best', 'outtmpl': outtmpl,
                'postprocessors': [{'key': 'FFmpegExtractAudio', 'preferredcodec': 'mp3',
                                    'preferredquality': quality or '192'}]}
    elif format == 'video':
        opts = {'quiet': True, 'format': 'bestvideo+bestaudio/best', 'outtmpl': outtmpl,
                'merge_output_format': 'mp4'}
    else:
        opts = {'quiet': True, 'format': format, 'outtmpl': outtmpl}
    with yt_dlp.YoutubeDL(opts) as ydl:
        info = ydl.extract_info(url, download=True)
        stem = Path(ydl.prepare_filename(info)).stem
    ext = 'mp3' if format == 'audio' else ('mp4' if format == 'video' else '*')
    candidates = sorted(save_dir.glob(f'{stem}*.{ext}'))
    return candidates[0] if candidates else save_dir / f'{stem}.{ext}'

In [ ]:
#| eval: false
p = download_yt('https://www.youtube.com/watch?v=aircAruvnKk', format='audio', save_dir='/tmp/fossick_test')
assert p.exists(), f"file not found: {p}"
assert p.suffix == '.mp3', f"expected .mp3, got {p.suffix}"
print(f"saved to: {p} ({p.stat().st_size // 1024} KB)")

## Install

In [ ]:
#| export
def repo_root() -> Path:
    'Find the root of the current git repository, or None if not in a repo.'
    return first((Path.cwd(), *Path.cwd().parents), lambda p: (p/'.git').exists())

def mv_skill_md(dry_run=True, dir=None) -> None:
    'Copy bundled SKILL.md to skill directories.'
    base = Path(__file__).parent if '__file__' in globals() else Path.cwd()
    if not (src := base.joinpath('SKILL.md')).exists(): return
    root = dir or repo_root() or '.'
    ts = [Path(root)/'.agents/skills/fossick/SKILL.md', Path('.claude/skills/fossick/SKILL.md')]
    if dry_run: print(f'Would copy {src} to: {list(map(str,ts))}')
    else:
        [p.mk_write(src.read_text(encoding='utf-8')) for p in ts]
        print(f'Installed → {list(map(str,ts))}')

In [ ]:
root = repo_root()
assert root is not None and (root/'.git').exists(), f"Expected git root, got {root}"
mv_skill_md(dry_run=True)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()